# GloFAS historical: quickstart

Download one day of global river discharge over the UK, via the
Copernicus CEMS Early Warning Data Store (EWDS).

See [`docs/glofas/README.md`](../docs/glofas/README.md) for the full
reference.

## Before you run this

1. Register at https://ewds.climate.copernicus.eu/ (**not** the main
   CDS - this is a separate platform).
2. Accept the GloFAS licence in your EWDS profile.
3. Copy your Personal Access Token from the EWDS profile page.
4. Export it in your shell before launching Jupyter:
   ```bash
   export EWDS_KEY=<your-token>
   ```
5. `pip install cdsapi xarray cfgrib eccodes matplotlib cartopy`

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
SYSTEM_VERSION = "version_4_0"
HYDROLOGICAL_MODEL = "lisflood"
PRODUCT_TYPE = "consolidated"
VARIABLE = "river_discharge_in_the_last_24_hours"
HYEAR = ["2020"]
HMONTH = ["12"]
HDAYS = ["01"]
BBOX = [60, -10, 50, 2]  # UK
OUTPUT_DIR = "../data/glofas"
OUTPUT_FILENAME = "glofas_quickstart.grib"
# ==================================================================

## Imports and environment check

In [ ]:
import os
import sys
from importlib.metadata import version
from pathlib import Path

import cdsapi
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["cdsapi", "xarray", "cfgrib", "matplotlib", "cartopy"]:
    try:
        print(f"{pkg:12} {version(pkg)}")
    except Exception:
        print(f"{pkg:12} (not installed)")

if not os.environ.get("EWDS_KEY"):
    raise RuntimeError("EWDS_KEY not set. Export your EWDS token first.")
print("\nEWDS_KEY detected in environment.")

## Submit the request to EWDS

Note the separate URL (`ewds.climate.copernicus.eu/api`) and the `h`-prefixed
date keys (`hyear`, `hmonth`, `hday`) which are GloFAS-specific.

In [ ]:
output_path = Path(OUTPUT_DIR) / OUTPUT_FILENAME
output_path.parent.mkdir(parents=True, exist_ok=True)

request = {
    "system_version": [SYSTEM_VERSION],
    "hydrological_model": [HYDROLOGICAL_MODEL],
    "product_type": [PRODUCT_TYPE],
    "variable": [VARIABLE],
    "hyear": HYEAR,
    "hmonth": HMONTH,
    "hday": HDAYS,
    "data_format": "grib2",
    "download_format": "unarchived",
    "area": BBOX,
}

client = cdsapi.Client(
    url="https://ewds.climate.copernicus.eu/api",
    key=os.environ["EWDS_KEY"],
)
client.retrieve("cems-glofas-historical", request).download(str(output_path))
print(f"Saved: {output_path} ({output_path.stat().st_size / 1e6:.2f} MB)")

## Open and inspect

GRIB2 output is opened via cfgrib. Discharge is reported in m3/s per grid
cell; only pixels on the LISFLOOD river network have meaningful values.

In [ ]:
ds = xr.open_dataset(output_path, engine="cfgrib")
print(ds)

## Plot a map of discharge

Because discharge values span many orders of magnitude (headwater m3/s vs
trunk-river thousands of m3/s), a log-scaled colourmap reads more clearly
than a linear one.

In [ ]:
from matplotlib.colors import LogNorm

var = "dis24" if "dis24" in ds.data_vars else list(ds.data_vars)[0]
dis = ds[var].squeeze()

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
dis.where(dis > 0).plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="Blues",
    norm=LogNorm(vmin=1, vmax=5000),
    cbar_kwargs={"label": "River discharge (m3/s, log)"},
)
ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
ax.set_title(f"GloFAS river discharge, UK, {HYEAR[0]}-{HMONTH[0]}-{HDAYS[0]}")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/glofas/README.md`](../docs/glofas/README.md)
- **Long time series at a gauging station**: find the grid cell nearest the
  station coordinates and extract a decades-long daily series to compare
  against observed streamflow.
- **Flood return periods**: fit a Generalised Extreme Value distribution to
  annual maximum daily discharge at each pixel.
- **Upstream catchment analysis**: use the auxiliary upstream-area layer
  from the GloFAS portal to identify the contributing basin for any grid
  cell.